<a href="https://colab.research.google.com/github/GonzaJapan/EcoLinker/blob/main/GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive をマウント
from google.colab import drive
# NumpyとPandasをインポート
import numpy as np
import pandas as pd
import math, random, torch, torch.nn as nn, torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import torch, torch.nn as nn
import numpy as np
import torch
from typing import List, Tuple, Optional, Dict
! pip install sentence-transformers torch

drive.mount('/content/drive')

Mounted at /content/drive


# 例

## 準備

In [ ]:
# 準備
import math, random, torch, torch.nn as nn, torch.nn.functional as F

# -----------------------------
# Toy data (P0..P2, Q0..Q3)
# -----------------------------
torch.manual_seed(123)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# process features (Np=3, dp_in=4)
Hp0 = torch.tensor([
    [0.8, 0.1, 0.0, 0.3],  # P0
    [0.4, 0.7, 0.2, 0.1],  # P1
    [0.2, 0.0, 0.9, 0.4],  # P2
], dtype=torch.float32, device=device)

# product features (Nq=4, dq_in=3)
Hq0 = torch.tensor([
    [0.6, 0.3, 0.1],  # Q0
    [0.2, 0.9, 0.1],  # Q1
    [0.1, 0.4, 0.8],  # Q2
    [0.7, 0.2, 0.0],  # Q3
], dtype=torch.float32, device=device)

Np, dp_in = Hp0.shape
Nq, dq_in = Hq0.shape

# Edges (forward only)
# consumes: product -> process (q -> p)
consumes_edges = [(0,0), (1,0), (1,1), (2,2)]
# produces: process -> product (p -> q)
produces_edges = [(0,2), (1,0), (1,3)]

## 正規化隣接行列

In [ ]:
# 列正規化の関数と正規化隣接行列
def row_norm(M: torch.Tensor) -> torch.Tensor:
    s = M.sum(dim=1, keepdim=True).clamp_min(1.0)
    return M / s

# A_cons: (|P| x |Q|) for Q->P
A_cons_pre = torch.zeros((Np, Nq), device=device)
for q, p in consumes_edges:
    A_cons_pre[p, q] += 1.0
A_cons = row_norm(A_cons_pre)

# A_prod: (|Q| x |P|) for P->Q
A_prod_pre = torch.zeros((Nq, Np), device=device)
for p, q in produces_edges:
    A_prod_pre[q, p] += 1.0
A_prod = row_norm(A_prod_pre)

# mirrors for encoder (rev_*): transpose + row-norm
A_rev_cons = row_norm(A_cons_pre.T)   # P->Q
A_rev_prod = row_norm(A_prod_pre.T)   # Q->P

In [ ]:
A_cons

tensor([[0.5000, 0.5000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000]], device='cuda:0')

In [ ]:
A_prod

tensor([[0., 1., 0.],
        [0., 0., 0.],
        [1., 0., 0.],
        [0., 1., 0.]], device='cuda:0')

In [ ]:
A_rev_cons

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000]], device='cuda:0')

In [ ]:
A_rev_prod

tensor([[0.0000, 0.0000, 1.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.5000],
        [0.0000, 0.0000, 0.0000, 0.0000]], device='cuda:0')

## モデル

In [ ]:
# モデル: 2-layer Hetero "SAGE-like" encoder + bilinear decoders
class TwoLayerHeteroSAGE(nn.Module):
    def __init__(self, d1=32, d2=32, dropout=0.1):
        super().__init__()
        self.dp_in, self.dq_in = dp_in, dq_in
        self.d1, self.d2 = d1, d2
        self.dropout = nn.Dropout(dropout)

        # Layer 1
        self.WP1     = nn.Linear(dp_in, d1, bias=True)   # process self
        self.WQ1     = nn.Linear(dq_in, d1, bias=True)   # product self
        self.Wcons1  = nn.Linear(dq_in, d1, bias=True)   # Q->P (consumes)
        self.Wrprod1 = nn.Linear(dq_in, d1, bias=True)   # Q->P (rev_produces)
        self.Wprod1  = nn.Linear(dp_in, d1, bias=True)   # P->Q (produces)
        self.Wrcons1 = nn.Linear(dp_in, d1, bias=True)   # P->Q (rev_consumes)

        # Layer 2
        self.WP2     = nn.Linear(d1, d2, bias=True)
        self.WQ2     = nn.Linear(d1, d2, bias=True)
        self.Wcons2  = nn.Linear(d1, d2, bias=True)
        self.Wrprod2 = nn.Linear(d1, d2, bias=True)
        self.Wprod2  = nn.Linear(d1, d2, bias=True)
        self.Wrcons2 = nn.Linear(d1, d2, bias=True)

        # Relation-specific bilinear heads
        self.R_cons  = nn.Linear(d2, d2, bias=False)  # for consumes (Q->P)
        self.R_prod  = nn.Linear(d2, d2, bias=False)  # for produces (P->Q)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=1.0)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, Hp, Hq):
        # Layer 1
        Hp1 = self.WP1(Hp) + A_cons @ self.Wcons1(Hq) + A_rev_prod @ self.Wrprod1(Hq)
        Hp1 = F.leaky_relu(Hp1, 0.1); Hp1 = self.dropout(Hp1)

        Hq1 = self.WQ1(Hq) + A_prod @ self.Wprod1(Hp) + A_rev_cons @ self.Wrcons1(Hp)
        Hq1 = F.leaky_relu(Hq1, 0.1); Hq1 = self.dropout(Hq1)

        # Layer 2
        Hp2 = self.WP2(Hp1) + A_cons @ self.Wcons2(Hq1) + A_rev_prod @ self.Wrprod2(Hq1)
        Hp2 = F.leaky_relu(Hp2, 0.1)
        Hq2 = self.WQ2(Hq1) + A_prod @ self.Wprod2(Hp1) + A_rev_cons @ self.Wrcons2(Hp1)
        Hq2 = F.leaky_relu(Hq2, 0.1)

        # L2 normalize (row-wise)
        Hp2 = Hp2 / (Hp2.norm(dim=1, keepdim=True) + 1e-9)
        Hq2 = Hq2 / (Hq2.norm(dim=1, keepdim=True) + 1e-9)
        return Hp2, Hq2

    # Decoders (schema-mask: only cross-type)
    def score_consumes(self, Hq, Hp, pairs):
        # pairs: list[(q,p)] for Q->P
        q = torch.tensor([qp[0] for qp in pairs], device=Hp.device)
        p = torch.tensor([qp[1] for qp in pairs], device=Hp.device)
        zq = Hq[q] @ self.R_cons.weight.T
        zp = Hp[p]
        return torch.sigmoid((zq * zp).sum(dim=-1))

    def score_produces(self, Hp, Hq, pairs):
        # pairs: list[(p,q)] for P->Q
        p = torch.tensor([pq[0] for pq in pairs], device=Hp.device)
        q = torch.tensor([pq[1] for pq in pairs], device=Hp.device)
        zp = Hp[p] @ self.R_prod.weight.T
        zq = Hq[q]
        return torch.sigmoid((zp * zq).sum(dim=-1))

## 訓練時ユーティリティ


*   sample_neg : 負例を作る
*   train :



In [ ]:
# 訓練時のユーティリティ関数
def sample_neg(existing_pairs, left_size, right_size, is_q_to_p):
    S = set(existing_pairs)
    out = []
    tries = 0
    target = max(1, len(existing_pairs))
    while len(out) < target and tries < 10000:
        if is_q_to_p:
            q = random.randrange(right_size)  # right side = Q
            p = random.randrange(left_size)   # left  side = P
            cand = (q, p)
        else:
            p = random.randrange(left_size)
            q = random.randrange(right_size)
            cand = (p, q)
        if cand not in S:
            out.append(cand)
        tries += 1
    return out

def train(model, epochs=600, lr=0.02, wd=1e-4):
    bce = nn.BCELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for ep in range(1, epochs+1):
        model.train(); opt.zero_grad()
        Hp2, Hq2 = model(Hp0, Hq0)

        # consumes (Q->P)
        neg_c = sample_neg(consumes_edges, Np, Nq, is_q_to_p=True)
        s_pos_c = model.score_consumes(Hq2, Hp2, consumes_edges)
        s_neg_c = model.score_consumes(Hq2, Hp2, neg_c)
        y_c = torch.cat([torch.ones_like(s_pos_c), torch.zeros_like(s_neg_c)])
        s_c = torch.cat([s_pos_c, s_neg_c])
        loss_c = bce(s_c, y_c)

        # produces (P->Q)
        neg_p = sample_neg(produces_edges, Np, Nq, is_q_to_p=False)
        s_pos_p = model.score_produces(Hp2, Hq2, produces_edges)
        s_neg_p = model.score_produces(Hp2, Hq2, neg_p)
        y_p = torch.cat([torch.ones_like(s_pos_p), torch.zeros_like(s_neg_p)])
        s_p = torch.cat([s_pos_p, s_neg_p])
        loss_p = bce(s_p, y_p)

        loss = loss_c + loss_p
        loss.backward(); opt.step()

        if ep % 100 == 0:
            print(f"epoch {ep:3d}  loss={loss.item():.4f}")

    return model


## インフラ ユーティリティ

In [ ]:
# インフラユーティリティ(Top-k)
@torch.no_grad()
def rank_produces(model, Hp2, Hq2, p_idx, topk=3):
    pairs = [(p_idx, q) for q in range(Hq2.size(0))]
    scores = model.score_produces(Hp2, Hq2, pairs).cpu()
    vals, idxs = torch.topk(scores, k=min(topk, len(pairs)))
    return [(int(q), float(v)) for q, v in zip(idxs, vals)]

@torch.no_grad()
def rank_consumes(model, Hp2, Hq2, q_idx, topk=3):
    pairs = [(q_idx, p) for p in range(Hp2.size(0))]
    scores = model.score_consumes(Hq2, Hp2, pairs).cpu()
    vals, idxs = torch.topk(scores, k=min(topk, len(pairs)))
    return [(int(p), float(v)) for p, v in zip(idxs, vals)]

@torch.no_grad()
def suggest_for_unknown_product(model, Hq_embed, Hp_embed, new_q, topk=3):
    # new_q: (dq_in,) tensor in device -> append to products, re-encode with same adjacencies (no edges to keep "unknown")
    global Hq0, Nq, A_cons, A_prod, A_rev_cons, A_rev_prod
    Hq_ext = torch.cat([Hq0, new_q.unsqueeze(0)], dim=0)
    # extend adjacencies with zero rows/cols (unknown not linked)
    A_cons_ext = torch.cat([A_cons, torch.zeros((Np,1), device=device)], dim=1)           # add new column for Q
    A_prod_ext = torch.cat([A_prod, torch.zeros((1,Np), device=device)], dim=0)           # add new row for Q
    A_rev_cons_ext = row_norm(A_cons_ext.T)                                               # rebuild mirrors
    A_rev_prod_ext = row_norm(A_prod_ext.T)

    # temporarily swap globals (simple demo)
    A_cons_bak, A_prod_bak = A_cons, A_prod
    A_rcons_bak, A_rprod_bak = A_rev_cons, A_rev_prod
    Hq0_bak = Hq0
    A_cons, A_prod, A_rev_cons, A_rev_prod = A_cons_ext, A_prod_ext, A_rev_cons_ext, A_rev_prod_ext
    Hq0 = Hq_ext

    Hp2, Hq2 = model(Hp0, Hq0)
    q_idx = Hq2.size(0) - 1
    # upstream producers: produces (p -> q*)
    up = rank_produces(model, Hp2, Hq2, p_idx=0, topk=1)  # dummy call to ensure shapes
    upstream = rank_consumes(model, Hp2, Hq2, q_idx, topk=topk)   # actually Q->P scores
    # downstream consumers: consumes (q* -> p)
    downstream = upstream  # same function already returns Q->P ranking

    # restore globals
    A_cons, A_prod, A_rev_cons, A_rev_prod = A_cons_bak, A_prod_bak, A_rcons_bak, A_rprod_bak
    Hq0 = Hq0_bak
    return upstream, downstream

@torch.no_grad()
def suggest_for_unknown_process(model, Hq_embed, Hp_embed, new_p, topk=3):
    global Hp0, Np, A_cons, A_prod, A_rev_cons, A_rev_prod
    Hp_ext = torch.cat([Hp0, new_p.unsqueeze(0)], dim=0)
    A_cons_ext = torch.cat([A_cons, torch.zeros((1, Nq), device=device)], dim=0)          # add new row for P
    A_prod_ext = torch.cat([A_prod, torch.zeros((Nq,1), device=device)], dim=1)           # add new column for P
    A_rev_cons_ext = row_norm(A_cons_ext.T)
    A_rev_prod_ext = row_norm(A_prod_ext.T)

    A_cons_bak, A_prod_bak = A_cons, A_prod
    A_rcons_bak, A_rprod_bak = A_rev_cons, A_rev_prod
    Hp0_bak = Hp0
    A_cons, A_prod, A_rev_cons, A_rev_prod = A_cons_ext, A_prod_ext, A_rev_cons_ext, A_rev_prod_ext
    Hp0 = Hp_ext

    Hp2, Hq2 = model(Hp0, Hq0)
    p_idx = Hp2.size(0) - 1
    # inputs: consumes (q -> p*)
    inputs = rank_consumes(model, Hp2, Hq2, q_idx=0, topk=1)  # dummy call
    inputs = [(q, s) for q, s in sorted([(q, model.score_consumes(Hq2, Hp2, [(q, p_idx)]).item())
                                         for q in range(Hq2.size(0))], key=lambda x: -x[1])[:topk]]
    # outputs: produces (p* -> q)
    outputs = [(q, s) for q, s in sorted([(q, model.score_produces(Hp2, Hq2, [(p_idx, q)]).item())
                                          for q in range(Hq2.size(0))], key=lambda x: -x[1])[:topk]]

    A_cons, A_prod, A_rev_cons, A_rev_prod = A_cons_bak, A_prod_bak, A_rcons_bak, A_rprod_bak
    Hp0 = Hp0_bak
    return inputs, outputs

## 訓練

In [ ]:
# Train & demo

model = TwoLayerHeteroSAGE(d1=16, d2=16, dropout=0.1).to(device)
model = train(model, epochs=600, lr=0.02, wd=1e-4)

model.eval()
Hp2, Hq2 = model(Hp0, Hq0)

# Show matrices (sigmoid applied)
S_prod = torch.sigmoid((Hp2 @ model.R_prod.weight.T) @ Hq2.T)  # P->Q
S_cons = torch.sigmoid((Hq2 @ model.R_cons.weight.T) @ Hp2.T)  # Q->P
print("\nProduces scores (rows=P, cols=Q):\n", S_prod.detach().cpu().numpy())
print("\nConsumes  scores (rows=Q, cols=P):\n", S_cons.detach().cpu().numpy())

# Ranking examples
for p in range(Np):
    top = sorted([(q, S_prod[p,q].item()) for q in range(Nq)], key=lambda x: -x[1])[:2]
    print(f"P{p} produces ->", top)
for q in range(Nq):
    top = sorted([(p, S_cons[q,p].item()) for p in range(Np)], key=lambda x: -x[1])[:2]
    print(f"Q{q} consumed by ->", top)

# Unknown product demo (random embed here;実務はテキスト埋め込み等)
new_q = torch.randn(dq_in, device=device)
up, down = suggest_for_unknown_product(model, Hq2, Hp2, new_q, topk=3)
print("\nUnknown Product q*: upstream producers (Q*->P):", up)
print("Unknown Product q*: downstream consumers (Q*->P):", down)

# Unknown process demo
new_p = torch.randn(dp_in, device=device)
inputs, outputs = suggest_for_unknown_process(model, Hq2, Hp2, new_p, topk=3)
print("\nUnknown Process p*: inputs (Q->P*):", inputs)
print("Unknown Process p*: outputs (P*->Q):", outputs)


epoch 100  loss=0.0490
epoch 200  loss=0.0207
epoch 300  loss=0.0130
epoch 400  loss=0.0141
epoch 500  loss=0.0170
epoch 600  loss=0.0103

Produces scores (rows=P, cols=Q):
 [[0.00222385 0.00842437 0.99768305 0.00769854]
 [0.99826247 0.01076888 0.00608665 0.9981845 ]
 [0.00928908 0.01105212 0.01248122 0.01075152]]

Consumes  scores (rows=Q, cols=P):
 [[0.9944665  0.00930828 0.00203208]
 [0.9975183  0.9960849  0.00578771]
 [0.00704244 0.00853698 0.9968874 ]
 [0.01321323 0.0040916  0.00907186]]
P0 produces -> [(2, 0.997683048248291), (1, 0.008424369618296623)]
P1 produces -> [(0, 0.9982624650001526), (3, 0.9981845021247864)]
P2 produces -> [(2, 0.01248121540993452), (1, 0.011052118614315987)]
Q0 consumed by -> [(0, 0.9944664835929871), (1, 0.009308279491961002)]
Q1 consumed by -> [(0, 0.9975183010101318), (1, 0.9960849285125732)]
Q2 consumed by -> [(2, 0.9968873858451843), (1, 0.008536981418728828)]
Q3 consumed by -> [(0, 0.01321322564035654), (2, 0.009071864187717438)]

Unknown Product 

# 製品とプロセスで実際にやってみよう

## 準備

In [ ]:
# 準備
torch.manual_seed(123)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dir = "/content/drive/MyDrive/LEC/しりょう/しりょう/"

# プロセス入出力
ProcessInputOutput = pd.read_excel(dir + "プロセス入出力_20250730.xlsx")
# プロセス
Process = pd.read_excel(dir + "プロセス_IDEA35確認済_20250709.xlsx")
# 製品
Product = pd.read_excel(dir + "製品_20250717.xlsx")


### エッジの作成

In [ ]:
# UUID→ID

# 例: 既存ノード
process_uuid_list = Process["*プロセスUUID"]
product_uuid_list = Product["*製品UUID"]
uuid2proc = dict(zip(Process["*プロセスUUID"], Process["プロセス名（日本語）"]))
uuid2prod = dict(zip(Product['*製品UUID'], Product['製品名（日本語）']))

# index化
proc2idx = {u:i for i,u in enumerate(process_uuid_list)}
prod2idx = {u:i for i,u in enumerate(product_uuid_list)}
idx2proc = {i:u for u,i in proc2idx.items()}
idx2prod = {i:u for u,i in prod2idx.items()}

In [ ]:
ProcessInputOutput_selected = ProcessInputOutput[["*プロセスUUID_LEC", "製品/基本フローUUID_LEC","*方向","流量", "単位コード", "フロータイプ"]]
ProcessInputOutput_selected.head()

,*プロセスUUID_LEC,製品/基本フローUUID_LEC,*方向,流量,単位コード,フロータイプ
0,42fc01e1-f085-4f8c-9c4d-840891a6f92f,bace4257-c770-41d4-8e10-86cee9d8cae9,入力,1.000000e+00,kg,中間フロー
1,42fc01e1-f085-4f8c-9c4d-840891a6f92f,5757186e-41f1-4af8-82e5-a9215daafcee,出力,1.000000e+00,kg,中間フロー
2,0f20deae-b898-41c1-b165-3953690d1c29,da5bcf4f-fa8b-45a7-98c9-4d4bf0a57c13,入力,1.776992e-10,kg,中間フロー
3,0f20deae-b898-41c1-b165-3953690d1c29,620a0ff5-fdf9-40d8-9cf4-0245008b2b85,入力,2.794397e-08,kg,中間フロー
4,0f20deae-b898-41c1-b165-3953690d1c29,32c427dc-f5f8-46be-9773-79d992c583cf,入力,1.137974e+00,MJ,中間フロー


各製品に関して「出力」となる入出力が1つしかないことが確認された

In [ ]:
Product_list = ProcessInputOutput["製品/基本フローUUID_LEC"].unique()
x = 0

for product in Product_list:
  x += 1
  output_count = ProcessInputOutput_selected[ProcessInputOutput_selected["製品/基本フローUUID_LEC"]=="da5bcf4f-fa8b-45a7-98c9-4d4bf0a57c13"]["*方向"].value_counts()["出力"]
  if output_count > 1:
    print(product)
  if x % 100 == 0:
    print(f"{x}個終わりました")


KeyboardInterrupt: 

In [ ]:
# プロセス入出力からエッジを作成
ProcessConsumes = ProcessInputOutput_selected[ProcessInputOutput_selected["*方向"]=="入力"]
ProcessProduces = ProcessInputOutput_selected[ProcessInputOutput_selected["*方向"]=="出力"]

# consumes: product -> process (q -> p)
consumes_edges = list(zip(ProcessConsumes['製品/基本フローUUID_LEC'], ProcessConsumes['*プロセスUUID_LEC']))
consumes_indexed_edges = [(prod2idx[u], proc2idx[v]) for u, v in consumes_edges]

for edge_pair in consumes_indexed_edges:
  product = edge_pair[0]
  process = edge_pair[1]
  if type(product) == str:
    print(f"製品{product}にUUIDが残っています")
  if type(process) == str:
    print(f"製品{process}にUUIDが残っています")

# produces: process -> product (p -> q)
produces_edges = list(zip(ProcessProduces['*プロセスUUID_LEC'], ProcessProduces['製品/基本フローUUID_LEC']))
produces_indexed_edges = [(proc2idx[u], prod2idx[v]) for u, v in produces_edges]

for edge_pair in produces_indexed_edges:
  product = edge_pair[1]
  process = edge_pair[0]
  if type(product) == str:
    print(f"製品{product}にUUIDが残っています")
  if type(process) == str:
    print(f"製品{process}にUUIDが残っています")

### 製品とプロセスをベクトルに変換する

プロセス

In [ ]:
# 1) 埋め込みモデル（多言語・日本語OKなものを選択）
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def build_process_text(row):
    # row は DataFrame の1行想定（辞書でも可）
    title = str(row['プロセス名（日本語）']).strip()
    scope = str(row['技術の範囲（日本語）']).strip()
    # テンプレで連結（順序は固定）
    return f"プロセス名: {title}\n技術の範囲: {scope}"

# 2) テキスト埋め込み（batched 推奨）
texts = [build_process_text(row) for _, row in Process.iterrows()]
emb_text = embedding_model.encode(texts, batch_size=64, normalize_embeddings=True)  # (Np, d_text) L2正規化
X_process = torch.tensor(emb_text, dtype=torch.float32)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
X_process.shape

torch.Size([9643, 384])

In [ ]:
Np, dp_in = X_process.shape

製品

本当は「麦わら, 入力, リマインダーフロー, JPN」となっているので、うまく分離させてちゃんと文章にしてベクトル化したいところ

In [ ]:
def build_product_text(row):
    # row は DataFrame の1行想定（辞書でも可）
    name = str(row['製品名（日本語）']).strip()

    # テンプレで連結（順序は固定）
    return f"製品: {name}"

# 2) テキスト埋め込み（batched 推奨）
texts = [build_product_text(row) for _, row in Product.iterrows()]
emb_text = embedding_model.encode(texts, batch_size=64, normalize_embeddings=True)  # (Np, d_text) L2正規化
X_product = torch.tensor(emb_text, dtype=torch.float32)

In [ ]:
X_product.shape

torch.Size([10159, 384])

In [ ]:
Nq, dq_in = X_product.shape

In [ ]:
X_process = X_process.to(device).float()
X_product = X_product.to(device).float()

## 正規隣接行列

疎行列を使って計算を行うことにする

In [ ]:
def build_sparse_adj(edges, num_rows, num_cols, device):
    """
    エッジリストから疎行列の隣接行列を構築し、行で正規化する関数
    """
    # エッジリストから行と列のインデックスを取得
    # PyTorchのCOO形式は (rows, cols) なので、タプルの2番目をrow、1番目をcolとして使う
    # consumes_indexed_edges は (q, p) の順なので、p, qの順に変換
    if not edges:
        # エッジが空の場合のゼロ行列を返す
        return torch.sparse_coo_tensor(torch.empty(2, 0), torch.empty(0), (num_rows, num_cols), device=device)

    rows = torch.tensor([edge[1] for edge in edges], dtype=torch.long, device=device)
    cols = torch.tensor([edge[0] for edge in edges], dtype=torch.long, device=device)

    # 重みはすべて1.0とする
    values = torch.ones(len(edges), device=device)

    # 疎行列を構築 (転置された形式)
    adj_pre = torch.sparse_coo_tensor(torch.stack([rows, cols], dim=0), values, (num_rows, num_cols))

    # 正規化
    row_sum = torch.sparse.sum(adj_pre, dim=1).to_dense()
    # 0除算を避ける
    row_sum[row_sum == 0] = 1e-12
    norm_values = values / row_sum[rows]
    adj_norm = torch.sparse_coo_tensor(torch.stack([rows, cols], dim=0), norm_values, (num_rows, num_cols))

    return adj_norm

# メインの処理
# ----------------------------------------------------------------------------------------------------

# consumes: Q->P (q, p)
A_cons = build_sparse_adj(consumes_indexed_edges, Np, Nq, device)

# produces: P->Q (p, q)
A_prod = build_sparse_adj(produces_indexed_edges, Nq, Np, device)

# mirrors for encoder (rev_*): transpose
# 疎行列の転置は非常に高速です
A_rev_cons = A_cons.T  # P->Q
A_rev_prod = A_prod.T  # Q->P

In [ ]:
A_cons      = A_cons.to(device)
A_prod      = A_prod.to(device)
A_rev_cons  = A_rev_cons.to(device)
A_rev_prod  = A_rev_prod.to(device)

## モデル

In [ ]:
# モデル: 2-layer Hetero "SAGE-like" encoder + bilinear decoders

# 特徴の shape から in_dim を決める
Np, dp_in = X_process.shape      # 例: (Np, 384)
Nq, dq_in = X_product.shape      # 例: (Nq, 384) など

# デバイス決定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TwoLayerHeteroSAGE(nn.Module):
    def __init__(self, d1=32, d2=32, dropout=0.1):
        super().__init__()
        self.dp_in, self.dq_in = dp_in, dq_in
        self.d1, self.d2 = d1, d2
        self.dropout = nn.Dropout(dropout)

        # Layer 1
        self.WP1     = nn.Linear(dp_in, d1, bias=True)   # process self
        self.WQ1     = nn.Linear(dq_in, d1, bias=True)   # product self
        self.Wcons1  = nn.Linear(dq_in, d1, bias=True)   # Q->P (consumes)
        self.Wrprod1 = nn.Linear(dq_in, d1, bias=True)   # Q->P (rev_produces)
        self.Wprod1  = nn.Linear(dp_in, d1, bias=True)   # P->Q (produces)
        self.Wrcons1 = nn.Linear(dp_in, d1, bias=True)   # P->Q (rev_consumes)

        # Layer 2
        self.WP2     = nn.Linear(d1, d2, bias=True)
        self.WQ2     = nn.Linear(d1, d2, bias=True)
        self.Wcons2  = nn.Linear(d1, d2, bias=True)
        self.Wrprod2 = nn.Linear(d1, d2, bias=True)
        self.Wprod2  = nn.Linear(d1, d2, bias=True)
        self.Wrcons2 = nn.Linear(d1, d2, bias=True)

        # Relation-specific bilinear heads
        self.R_cons  = nn.Linear(d2, d2, bias=False)  # for consumes (Q->P)
        self.R_prod  = nn.Linear(d2, d2, bias=False)  # for produces (P->Q)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=1.0)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, X_process, X_product):
        # Layer 1
        Hp1 = self.WP1(X_process) + A_cons @ self.Wcons1(X_product) + A_rev_prod @ self.Wrprod1(X_product)
        Hp1 = F.leaky_relu(Hp1, 0.1); Hp1 = self.dropout(Hp1)

        Hq1 = self.WQ1(X_product) + A_prod @ self.Wprod1(X_process) + A_rev_cons @ self.Wrcons1(X_process)
        Hq1 = F.leaky_relu(Hq1, 0.1); Hq1 = self.dropout(Hq1)

        # Layer 2
        Hp2 = self.WP2(Hp1) + A_cons @ self.Wcons2(Hq1) + A_rev_prod @ self.Wrprod2(Hq1)
        Hp2 = F.leaky_relu(Hp2, 0.1)
        Hq2 = self.WQ2(Hq1) + A_prod @ self.Wprod2(Hp1) + A_rev_cons @ self.Wrcons2(Hp1)
        Hq2 = F.leaky_relu(Hq2, 0.1)

        # L2 normalize (row-wise)
        Hp2 = Hp2 / (Hp2.norm(dim=1, keepdim=True) + 1e-9)
        Hq2 = Hq2 / (Hq2.norm(dim=1, keepdim=True) + 1e-9)
        return Hp2, Hq2

    # Decoders (schema-mask: only cross-type)
    def score_consumes(self, Hq, Hp, pairs):
        # pairs: list[(q,p)] for Q->P
        q = torch.tensor([qp[0] for qp in pairs], device=device)
        p = torch.tensor([qp[1] for qp in pairs], device=device)
        zq = Hq[q] @ self.R_cons.weight.T
        zp = Hp[p]
        return torch.sigmoid((zq * zp).sum(dim=-1))

    def score_produces(self, Hp, Hq, pairs):
        # pairs: list[(p,q)] for P->Q
        p = torch.tensor([pq[0] for pq in pairs], device=device)
        q = torch.tensor([pq[1] for pq in pairs], device=device)
        zp = Hp[p] @ self.R_prod.weight.T
        zq = Hq[q]
        return torch.sigmoid((zp * zq).sum(dim=-1))

### 訓練ユーティリティ

In [ ]:
# ========= 基本ユーティリティ =========
def row_norm(M: torch.Tensor) -> torch.Tensor:
    return M / (M.sum(dim=1, keepdim=True).clamp_min(1.0))

def split_edges_85_15(edges: List[Tuple[int,int]], val_ratio: float = 0.15, seed: int = 42):
    rng = random.Random(seed)
    idx = list(range(len(edges)))
    rng.shuffle(idx)
    k = int(len(idx) * (1 - val_ratio))
    train_idx = idx[:k]
    val_idx   = idx[k:]
    edges_train = [edges[i] for i in train_idx]
    edges_val   = [edges[i] for i in val_idx]
    return edges_train, edges_val

def build_adj_from_edges(Np: int, Nq: int,
                         consumes_edges_train: List[Tuple[int,int]], # (q,p) Q->P
                         produces_edges_train: List[Tuple[int,int]], # (p,q) P->Q
                         device):
    # 生の0/1隣接（密で作る簡易版。大規模なら疎に置き換え）
    B_cons = torch.zeros((Np, Nq), device=device)
    for (q, p) in consumes_edges_train:
        B_cons[p, q] = 1.0
    B_prod = torch.zeros((Nq, Np), device=device)
    for (p, q) in produces_edges_train:
        B_prod[q, p] = 1.0

    # 行正規化（mean集約）
    A_cons = row_norm(B_cons)       # Q->P（受け手=process）
    A_prod = row_norm(B_prod)       # P->Q（受け手=product）
    A_rev_cons = row_norm(B_cons.T) # P->Q
    A_rev_prod = row_norm(B_prod.T) # Q->P
    return A_cons, A_prod, A_rev_cons, A_rev_prod

def sample_neg(existing_pairs: List[Tuple[int,int]], left_size: int, right_size: int, is_q_to_p: bool, num: int, seed: int = 0):
    S = set(existing_pairs)
    out = []
    rng = random.Random(seed)
    tries = 0
    max_tries = max(10000, num * 10)
    while len(out) < num and tries < max_tries:
        if is_q_to_p:
            q = rng.randrange(right_size)  # product idx
            p = rng.randrange(left_size)   # process idx
            cand = (q, p)
        else:
            p = rng.randrange(left_size)   # process idx
            q = rng.randrange(right_size)  # product idx
            cand = (p, q)
        if cand not in S:
            out.append(cand)
        tries += 1
    if len(out) < num:
        # 足りない場合は重複許容で埋める（最後の手段）
        while len(out) < num:
            if is_q_to_p:
                out.append((rng.randrange(right_size), rng.randrange(left_size)))
            else:
                out.append((rng.randrange(left_size), rng.randrange(right_size)))
    return out

def bce_link_loss(model, Hp2, Hq2,
                  pos_c: List[Tuple[int,int]], # (q,p)
                  pos_p: List[Tuple[int,int]], # (p,q)
                  neg_mult: int = 1, seed: int = 0):
    Np = Hp2.size(0); Nq = Hq2.size(0)
    # consumes (Q->P)
    neg_c = sample_neg(pos_c, left_size=Np, right_size=Nq, is_q_to_p=True,
                       num=max(1, len(pos_c)*neg_mult), seed=seed)
    s_pos_c = model.score_consumes(Hq2, Hp2, pos_c)
    s_neg_c = model.score_consumes(Hq2, Hp2, neg_c)
    y_c = torch.cat([torch.ones_like(s_pos_c), torch.zeros_like(s_neg_c)])
    s_c = torch.cat([s_pos_c, s_neg_c])
    loss_c = F.binary_cross_entropy(s_c, y_c)

    # produces (P->Q)
    neg_p = sample_neg(pos_p, left_size=Np, right_size=Nq, is_q_to_p=False,
                       num=max(1, len(pos_p)*neg_mult), seed=seed+1)
    s_pos_p = model.score_produces(Hp2, Hq2, pos_p)
    s_neg_p = model.score_produces(Hp2, Hq2, neg_p)
    y_p = torch.cat([torch.ones_like(s_pos_p), torch.zeros_like(s_neg_p)])
    s_p = torch.cat([s_pos_p, s_neg_p])
    loss_p = F.binary_cross_entropy(s_p, y_p)

    return loss_c + loss_p

In [ ]:
def train_with_val(model,
                   X_process: torch.Tensor, X_product: torch.Tensor,
                   consumes_edges: List[Tuple[int,int]],  # 全エッジ（Q->P）
                   produces_edges: List[Tuple[int,int]],  # 全エッジ（P->Q）
                   epochs: int = 200, lr: float = 0.02, wd: float = 1e-4,
                   val_ratio: float = 0.15, neg_mult: int = 1,
                   seed: int = 42, verbose: bool = True):
    dev = X_process.device
    Np, Nq = X_process.size(0), X_product.size(0)

    # 1) train/val 分割（85/15）
    cons_tr, cons_val = split_edges_85_15(consumes_edges, val_ratio, seed)
    prod_tr, prod_val = split_edges_85_15(produces_edges, val_ratio, seed)

    # 2) 学習グラフ（trainのみ）を構築
    A_cons, A_prod, A_rev_cons, A_rev_prod = build_adj_from_edges(Np, Nq, cons_tr, prod_tr, dev)

    # 3) optimizer
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    # 4) 学習ループ
    history = []
    best_state = None
    best_val = float('inf')

    for ep in range(1, epochs+1):
        model.train()
        # 埋め込み（trainグラフ）
        if hasattr(model, "forward_with_adj"):
            Hp2, Hq2 = model.forward_with_adj(X_process, X_product, A_cons, A_prod, A_rev_cons, A_rev_prod)
        else:
            Hp2, Hq2 = model(X_process, X_product)

        # train loss（trainエッジで）
        loss = bce_link_loss(model, Hp2, Hq2, cons_tr, prod_tr, neg_mult=neg_mult, seed=seed+ep)
        opt.zero_grad()
        loss.backward()
        opt.step()

        # 検証（valエッジ、埋め込みは学習グラフのまま）
        model.eval()
        with torch.no_grad():
            if hasattr(model, "forward_with_adj"):
                Hp2v, Hq2v = model.forward_with_adj(X_process, X_product, A_cons, A_prod, A_rev_cons, A_rev_prod)
            else:
                Hp2v, Hq2v = model(X_process, X_product)
            val_loss = bce_link_loss(model, Hp2v, Hq2v, cons_val, prod_val, neg_mult=neg_mult, seed=seed)

        history.append({"epoch": ep, "train_loss": float(loss.item()), "val_loss": float(val_loss.item())})

        # ベスト更新（val 最小）
        if val_loss.item() < best_val:
            best_val = val_loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if verbose and (ep % 20 == 0 or ep == 1):
            print(f"[{ep:4d}] train_loss={loss.item():.4f}  val_loss={val_loss.item():.4f}")

    # 5) ベスト重みを復元
    if best_state is not None:
        model.load_state_dict(best_state)

    # 最終の埋め込み（trainグラフ）も返すと便利
    model.eval()
    with torch.no_grad():
        if hasattr(model, "forward_with_adj"):
            Hp2_final, Hq2_final = model.forward_with_adj(X_process, X_product, A_cons, A_prod, A_rev_cons, A_rev_prod)
        else:
            Hp2_final, Hq2_final = model(X_process, X_product)

    # 返り値：モデル、履歴、分割結果（評価再現用）、最終埋め込み
    split = {
        "consumes_train": cons_tr, "consumes_val": cons_val,
        "produces_train": prod_tr, "produces_val": prod_val
    }
    return model, history, split, (Hp2_final, Hq2_final)

In [ ]:
@torch.no_grad()
def _topk_with_mask(scores: torch.Tensor, k: int, forbid_idx: Optional[List[int]] = None):
    """
    scores: (N,) のスコア。forbid_idx に含まれる候補は -inf にして除外。
    """
    s = scores.clone()
    if forbid_idx:
        s[torch.tensor(forbid_idx, dtype=torch.long, device=s.device)] = float("-inf")
    k = min(k, s.numel())
    vals, idxs = torch.topk(s, k=k)
    return [(int(i), float(v)) for i, v in zip(idxs, vals)]

# ========= Top-k: 既存ノードに対するランキング =========
@torch.no_grad()
def rank_produces(model, Hp2: torch.Tensor, Hq2: torch.Tensor, p_idx: int, topk=10,
                  tau: float = 1.0, forbid_q: Optional[List[int]] = None) -> List[Tuple[int, float]]:
    """
    P->Q の Top-k。双線形を行列積で一括計算。
    """
    dev = Hp2.device
    # s(q) = σ( (Hp2[p] W) · Hq2[q] )
    W = model.R_prod.weight    # (d,d)
    zp = (Hp2[p_idx] @ W.T).unsqueeze(0)      # (1,d)
    scores = torch.sigmoid( (zp @ Hq2.T).squeeze(0) / tau )  # (Nq,)
    return _topk_with_mask(scores, topk, forbid_q)

@torch.no_grad()
def rank_consumes(model, Hp2: torch.Tensor, Hq2: torch.Tensor, q_idx: int, topk=10,
                  tau: float = 1.0, forbid_p: Optional[List[int]] = None) -> List[Tuple[int, float]]:
    """
    Q->P の Top-k。
    """
    dev = Hq2.device
    W = model.R_cons.weight
    zq = (Hq2[q_idx] @ W.T).unsqueeze(0)      # (1,d)
    scores = torch.sigmoid( (zq @ Hp2.T).squeeze(0) / tau )  # (Np,)
    return _topk_with_mask(scores, topk, forbid_p)

import torch
from typing import List, Tuple

@torch.no_grad()
def encode_unknown_product(model, new_q_feat: torch.Tensor) -> torch.Tensor:
    """隣接不要で q* の最終埋め込み h_q^* を作る"""
    dev = next(model.parameters()).device
    h1 = torch.relu(model.WQ1(new_q_feat.to(dev)))
    h2 = torch.relu(model.WQ2(h1))
    h2 = h2 / (h2.norm() + 1e-9)
    return h2  # (d,)

@torch.no_grad()
def encode_unknown_process(model, new_p_feat: torch.Tensor) -> torch.Tensor:
    dev = next(model.parameters()).device
    h1 = torch.relu(model.WP1(new_p_feat.to(dev)))
    h2 = torch.relu(model.WP2(h1))
    h2 = h2 / (h2.norm() + 1e-9)
    return h2  # (d,)

@torch.no_grad()
def suggest_for_unknown_product_noextend(
    model, Hp2: torch.Tensor, Hq2: torch.Tensor, new_q_feat: torch.Tensor, topk: int = 10, tau: float = 1.0
) -> Tuple[List[Tuple[int,float]], List[Tuple[int,float]]]:
    """
    既存 Hp2/Hq2 をそのまま使い、q* の埋め込みだけ単体で作って Top-k を返す。
    戻り値: (上流:誰が作る? Pリスト, 下流:誰に消費される? Pリスト)
    """
    dev = Hp2.device
    hq_star = encode_unknown_product(model, new_q_feat)           # (d,)
    # 上流: produces (P -> q*)
    zp_all = Hp2 @ model.R_prod.weight.T                          # (Np,d)
    scores_up = torch.sigmoid((zp_all @ hq_star) / tau)           # (Np,)
    vals, idxs = torch.topk(scores_up, k=min(topk, scores_up.numel()))
    upstream = [(uuid2proc[idx2proc[int(i)]], float(v)) for i, v in zip(idxs, vals)]

    # 下流: consumes (q* -> P)
    zq_star = hq_star @ model.R_cons.weight.T                     # (d,)
    scores_dn = torch.sigmoid((Hp2 @ zq_star) / tau)              # (Np,)
    vals, idxs = torch.topk(scores_dn, k=min(topk, scores_dn.numel()))
    downstream = [(uuid2proc[idx2proc[int(i)]], float(v)) for i, v in zip(idxs, vals)]
    return upstream, downstream

@torch.no_grad()
def suggest_for_unknown_process_noextend(
    model, Hp2: torch.Tensor, Hq2: torch.Tensor, new_p_feat: torch.Tensor, topk: int = 10, tau: float = 1.0
) -> Tuple[List[Tuple[int,float]], List[Tuple[int,float]]]:
    """
    戻り値: (入力:何を消費? Qリスト, 出力:何を生産? Qリスト)
    """
    dev = Hq2.device
    hp_star = encode_unknown_process(model, new_p_feat)           # (d,)
    # 入力: consumes (Q -> p*)
    zq_all = Hq2 @ model.R_cons.weight.T                          # (Nq,d)
    scores_in = torch.sigmoid((zq_all @ hp_star) / tau)           # (Nq,)
    vals, idxs = torch.topk(scores_in, k=min(topk, scores_in.numel()))
    inputs = [(uuid2prod[idx2prod[int(i)]], float(v)) for i, v in zip(idxs, vals)]

    # 出力: produces (p* -> Q)
    zp_star = hp_star @ model.R_prod.weight.T                     # (d,)
    scores_out = torch.sigmoid((Hq2 @ zp_star) / tau)             # (Nq,)
    vals, idxs = torch.topk(scores_out, k=min(topk, scores_out.numel()))
    outputs = [(uuid2prod[idx2prod[int(i)]], float(v)) for i, v in zip(idxs, vals)]
    return inputs, outputs



## 訓練

In [ ]:
# Train
model = TwoLayerHeteroSAGE(d1=256, d2=256, dropout=0.1).to(device)
model, history, split, (Hp2, Hq2) = train_with_val(
    model,
    X_process.to(device), X_product.to(device),
    consumes_indexed_edges,  # (q,p) indexリスト
    produces_indexed_edges,  # (p,q) indexリスト
    epochs=500, lr=0.02, wd=1e-4,
    val_ratio=0.15, neg_mult=1, seed=42, verbose=True
)

[   1] train_loss=1.4035  val_loss=1.3135
[  20] train_loss=0.8769  val_loss=0.8740
[  40] train_loss=0.6965  val_loss=0.7066
[  60] train_loss=0.7880  val_loss=0.8065
[  80] train_loss=0.5816  val_loss=0.5672
[ 100] train_loss=0.4646  val_loss=0.4672
[ 120] train_loss=0.3301  val_loss=0.3421
[ 140] train_loss=0.7354  val_loss=0.5678
[ 160] train_loss=0.4075  val_loss=0.4141
[ 180] train_loss=0.3910  val_loss=0.3693
[ 200] train_loss=0.3606  val_loss=0.3601
[ 220] train_loss=0.3372  val_loss=0.3535
[ 240] train_loss=0.3364  val_loss=0.3382
[ 260] train_loss=0.3094  val_loss=0.3199
[ 280] train_loss=0.3013  val_loss=0.3138
[ 300] train_loss=0.3700  val_loss=0.3905
[ 320] train_loss=0.3139  val_loss=0.3265
[ 340] train_loss=0.3033  val_loss=0.3004
[ 360] train_loss=0.3168  val_loss=0.3148
[ 380] train_loss=0.3023  val_loss=0.3348
[ 400] train_loss=0.2763  val_loss=0.2773
[ 420] train_loss=0.5023  val_loss=0.4677
[ 440] train_loss=0.3404  val_loss=0.3416
[ 460] train_loss=0.3039  val_loss

In [ ]:
# Demo
#Hp2, Hq2 = (Hp2, Hq2)

# Show matrices (sigmoid applied)
S_prod = torch.sigmoid((Hp2 @ model.R_prod.weight.T) @ Hq2.T)  # P->Q
S_cons = torch.sigmoid((Hq2 @ model.R_cons.weight.T) @ Hp2.T)  # Q->P
print("\nProduces scores (rows=P, cols=Q):\n", S_prod.detach().cpu().numpy())
print("\nConsumes  scores (rows=Q, cols=P):\n", S_cons.detach().cpu().numpy())


Produces scores (rows=P, cols=Q):
 [[1.8880374e-03 3.5139977e-04 1.2507492e-01 ... 8.8054547e-03
  9.9951148e-02 3.0862144e-03]
 [4.6570200e-04 1.5748162e-03 9.5506053e-04 ... 1.8164112e-01
  1.1839934e-03 1.6575895e-01]
 [3.9689168e-03 1.8384684e-03 9.2943693e-03 ... 7.3043525e-02
  2.1470828e-01 2.6427478e-02]
 ...
 [1.7019683e-02 3.9350633e-03 5.4468215e-01 ... 5.1946228e-04
  1.1368109e-01 3.8737623e-04]
 [2.5205656e-03 2.4863832e-02 1.4353598e-03 ... 3.0780785e-02
  2.7832508e-04 8.0620557e-02]
 [3.8337638e-03 3.3162106e-02 2.2776243e-03 ... 2.3734249e-02
  3.8162083e-04 6.5904237e-02]]

Consumes  scores (rows=Q, cols=P):
 [[0.06476381 0.0169746  0.00507479 ... 0.04746341 0.00781019 0.00941144]
 [0.01119605 0.03647219 0.00207369 ... 0.00770871 0.02964253 0.0330966 ]
 [0.24530016 0.00679702 0.07295804 ... 0.3668092  0.00511186 0.00730534]
 ...
 [0.00295257 0.03586024 0.04007876 ... 0.00285473 0.06821436 0.08032193]
 [0.05714655 0.00287609 0.10793962 ... 0.11610089 0.00352299 0.005

未知の製品

In [ ]:
def build_process_text_from_fields(title: str) -> str:
    title = (title or "").strip()
    return f"製品: {title}"

def encode_unknown_product_text(model: embedding_model, title: str) -> torch.Tensor:
  """
  未知プロセス（title, scope）→ ベクトル（torch.float32, shape=(d_text,)）
  既存の学習で使った埋め込みと同じ前処理・同じ正規化を踏襲。
  """
  text = build_process_text_from_fields(title)
  emb = model.encode([text], batch_size=1, normalize_embeddings=True)  # (1, d_text), L2正規化済み
  vec = torch.tensor(emb[0], dtype=torch.float32)                      # (d_text,)
  return vec

In [ ]:
# 未知 product
title = "チタン採掘"
new_q_feat = encode_unknown_product_text(embedding_model, title=title)
new_q_feat = new_q_feat.to(device)
up, down = suggest_for_unknown_product_noextend(model, Hp2, Hq2, new_q_feat, topk=10)

print(f" inputs (プロセス->{title}):, {up}")
print(f" inputs ({title}->プロセス):, {down}")

 inputs (プロセス->チタン鉱石, MOZ):, [('S工場の建設, GLO', 0.15104269981384277), ('RC工場の建設, GLO', 0.15051871538162231), ('木造工場の建設, GLO', 0.1411048322916031), ('一般廃棄物ごみ処分, 4桁, GLO', 0.13544133305549622), ('RC工場の建設, JPN', 0.11857400089502335), ('S工場の建設, JPN', 0.11078877747058868), ('一般廃棄物ごみ処分, 4桁, JPN', 0.10429742187261581), ('自然科学研究機関, 国公立, GLO', 0.09711242467164993), ('木造工場の建設, JPN', 0.09685742855072021), ('産廃処理, 動植物性残渣, GLO', 0.08744599670171738)]
 inputs (チタン鉱石, MOZ->プロセス):, [('RC工場の建設, JPN', 0.530965268611908), ('S工場の建設, JPN', 0.509811520576477), ('木造工場の建設, JPN', 0.4815090596675873), ('S工場の建設, GLO', 0.47087568044662476), ('RC工場の建設, GLO', 0.4702437222003937), ('木造工場の建設, GLO', 0.4619193971157074), ('SRC工場の建設, GLO', 0.371509850025177), ('農林関係公共事業, GLO', 0.28417688608169556), ('工業用水道, 消費型使用水, JPN', 0.281730055809021), ('保健衛生, 国公立, GLO', 0.28142473101615906)]
